In [1]:
# =========================================
# 1. INSTALL
# =========================================
# pip install sentence-transformers transformers rapidfuzz torch

# =========================================
# 2. IMPORTS
# =========================================
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
from rapidfuzz import fuzz
import re


d:\dream_machine\AI-content-moderator\env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# =========================================
# 3. LOAD MODELS
# =========================================

# Grooming (semantic)
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Toxicity (reuse your existing)
toxicity_model = pipeline(
    "text-classification",
    model="unitary/unbiased-toxic-roberta",
    return_all_scores=True
)

# Explicit / unsafe (light usage)
moderation_model = pipeline(
    "text-classification",
    model="KoalaAI/Text-Moderation",
    return_all_scores=True
)



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6725.51it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 18519.73it/s]
RobertaForSequenceClassification LOAD REPORT from: unitary/unbiased-toxic-roberta
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 200/200 [00:00<00:00, 4985.21it/s]
DebertaForSequenceClassification LOAD REPORT from: KoalaAI/Text-Moderation
Key                             | Status    

In [3]:
# =========================================
# 4. NORMALIZATION
# =========================================
def normalize_text(text):
    text = text.lower()

    replacements = {
        "0": "o", "1": "i", "3": "e",
        "@": "a", "$": "s"
    }

    for k, v in replacements.items():
        text = text.replace(k, v)

    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text



In [4]:
# =========================================
# 5. GROOMING PATTERNS
# =========================================
GROOMING_PATTERNS = [
    "let's keep this between us",
    "don't tell your parents",
    "this is our secret",
    "you are special to me",
    "i understand you better than others",
    "we should meet privately",
    "just the two of us",
    "dont tell anyone",
    "keep this secret",
    "come alone",
    "meet alone"
]

pattern_embeddings = embedder.encode(GROOMING_PATTERNS)



In [5]:
# =========================================
# 6. SEMANTIC SCORE
# =========================================
def semantic_score(messages):
    text = normalize_text(" ".join(messages[-5:]))
    emb = embedder.encode(text)
    scores = util.cos_sim(emb, pattern_embeddings)
    return float(scores.max())



In [6]:
# =========================================
# 7. RULE SCORE
# =========================================
def rule_score(messages):
    score = 0

    for msg in messages:
        msg = normalize_text(msg)

        if "secret" in msg or "dont tell" in msg:
            score += 2

        if "meet" in msg and "alone" in msg:
            score += 2

        if "private" in msg:
            score += 1.5

        if "special" in msg or "trust me" in msg:
            score += 1

    return min(score / 5, 1)



In [7]:
# =========================================
# 8. FUZZY SCORE
# =========================================
def fuzzy_score(text):
    phrases = ["dont tell", "secret", "meet alone"]
    score = 0

    for p in phrases:
        score = max(score, fuzz.partial_ratio(text, p) / 100)

    return score * 0.5



In [8]:
# =========================================
# 9. CONTEXT PENALTY
# =========================================
def context_penalty(text):
    SAFE_WORDS = ["recipe", "project", "assignment", "team", "meeting"]

    penalty = 0
    for word in SAFE_WORDS:
        if word in text:
            penalty += 0.2

    # override if strong grooming signal exists
    if "meet alone" in text or "dont tell" in text:
        return 0

    return min(penalty, 0.3)



In [9]:
def gray_signal_boost(messages):
    text = normalize_text(" ".join(messages))

    # Gray patterns (not outright grooming, but suspicious)
    if "meet" in text and "alone" in text:
        return 0.15

    if "private" in text:
        return 0.1

    return 0

In [10]:
# =========================================
# 10. CRITICAL OVERRIDE
# =========================================
def critical_override(messages):
    text = normalize_text(" ".join(messages))

    if "dont tell" in text and ("meet" in text or "alone" in text):
        return True

    if "secret" in text and "private" in text:
        return True

    return False



In [11]:
def detect_grooming(messages):
    text = normalize_text(" ".join(messages))

    sem = semantic_score(messages)
    rule = rule_score(messages)
    fuzzy = fuzzy_score(text)
    penalty = context_penalty(text)
    gray_boost = gray_signal_boost(messages)

    final_score = (
        0.6 * sem +
        0.3 * rule +
        0.1 * fuzzy +
        gray_boost          # 🔥 NEW
    )

    final_score = max(0, final_score - penalty)

    if critical_override(messages):
        final_score = max(final_score, 0.85)

    if final_score > 0.7:
        level = "HIGH"
    elif final_score > 0.4:
        level = "MEDIUM"
    else:
        level = "LOW"

    return {
        "risk_score": round(final_score, 3),
        "level": level,
        "signals": {
            "semantic": round(sem, 3),
            "rule": round(rule, 3),
            "fuzzy": round(fuzzy, 3),
            "gray_boost": gray_boost
        }
    }

In [12]:
# =========================================
# 12. TOXICITY DETECTOR
# =========================================
def get_toxicity_score(text):
    results = toxicity_model(text)

    if isinstance(results, list):
        results = results[0]

    if isinstance(results, dict):
        return results["score"]

    return max(r['score'] for r in results)

# =========================================
# 13. EXPLICIT CONTENT DETECTOR
# =========================================
def get_explicit_score(text):
    results = moderation_model(text)

    if isinstance(results, list):
        results = results[0]

    if isinstance(results, dict):
        return results.get("score", 0)

    scores = {r['label']: r['score'] for r in results}
    return scores.get("sexual", 0)



In [13]:
# =========================================
# 14. FINAL SAFETY PIPELINE
# =========================================
def analyze_conversation(messages):
    text = " ".join(messages)

    grooming = detect_grooming(messages)
    toxicity = get_toxicity_score(text)
    explicit = get_explicit_score(text)

    return {
        "grooming": grooming,
        "toxicity_score": round(toxicity, 3),
        "explicit_score": round(explicit, 3)
    }



In [16]:
# =========================================
# 15. TEST
# =========================================
tests = [
    ["hey how was your day?", "see you tomorrow"],
    ["you are special to me", "dont tell anyone"],
    ["So this is the secret recipe", "lets meet at a cafe alone"],
    ["d0nt t3ll any0ne", "lets m33t al0ne"],
    ["Lets fuck"]
]

tests = [
    ["you are really special to me", "i dont say this to anyone"],
    ["i trust you more than anyone else", "you understand me"],
    ["you are different from others", "i feel close to you"],
    ["i feel like we have a connection", "stronger than others"],
    ["you can tell me anything", "i wont judge you"]
]

for t in tests:
    print("Messages:", t)
    print(analyze_conversation(t))
    print("-" * 60)

Messages: ['you are really special to me', 'i dont say this to anyone']
{'grooming': {'risk_score': 0.578, 'level': 'MEDIUM', 'signals': {'semantic': 0.81, 'rule': 0.2, 'fuzzy': 0.316, 'gray_boost': 0}}, 'toxicity_score': 0.0, 'explicit_score': 0.951}
------------------------------------------------------------
Messages: ['i trust you more than anyone else', 'you understand me']
{'grooming': {'risk_score': 0.417, 'level': 'MEDIUM', 'signals': {'semantic': 0.649, 'rule': 0.0, 'fuzzy': 0.278, 'gray_boost': 0}}, 'toxicity_score': 0.001, 'explicit_score': 0.961}
------------------------------------------------------------
Messages: ['you are different from others', 'i feel close to you']
{'grooming': {'risk_score': 0.355, 'level': 'LOW', 'signals': {'semantic': 0.537, 'rule': 0.0, 'fuzzy': 0.333, 'gray_boost': 0}}, 'toxicity_score': 0.001, 'explicit_score': 0.981}
------------------------------------------------------------
Messages: ['i feel like we have a connection', 'stronger than othe